# 初试 TVM

配置工具：

```bash
conda create -n py312 python=3.12
conda activate py312
git clone https://github.com/xinetzone/tvm
cd tvm/xinetzone
pip install .[doc,dev] -i https://pypi.tuna.tsinghua.edu.cn/simple
pip install pickleshare -i https://pypi.tuna.tsinghua.edu.cn/simple # 用于 jupyter notebook 魔法指令
```

配置环境：

```bash
python -m invoke init
python -m invoke config --cuda
python -m invoke make
```


配置前端框架(包含 CUDA+CUDNN)：

```bash
conda install pytorch torchvision torchaudio cuda-nvcc cudnn  cudatoolkit pytorch-cuda=12.4 -c pytorch -c nvidia
pip install tensorflow -i https://pypi.tuna.tsinghua.edu.cn/simple
pip install onnx onnxruntime-gpu -i https://pypi.tuna.tsinghua.edu.cn/simple
# caffe 依赖
pip install scikit-image -i https://pypi.tuna.tsinghua.edu.cn/simple
pip install protobuf==3.20.3 -i https://pypi.tuna.tsinghua.edu.cn/simple
```


加载 TVM 环境：

In [1]:
%cd ..
import set_env
from d2py.utils.file import mkdir
root_dir = ".temp" # 用于存储缓存内容
mkdir(f"{root_dir}/logs") # 构建日志记录目录

/media/pc/data/lxw/ai/tvm-book/tests/book/doc


## 构建数据集

In [2]:
from pathlib import Path
from copy import deepcopy
import numpy as np
from tqdm import tqdm
from utils import preprocessing

ENV = {
    "model_type": "onnx",
    "input_name": "x",
    "channel": 3,
    "height": 256, 
    "width": 192,
    "mode": "RGB", # 输入图片格式
    "mean": (123.675, 116.280, 103.530),
    "std": (58.395, 57.120, 57.375)
}

def calibrateset(calibrate_num=2, data_dir="/media/pc/data/lxw/home/data/coco/train2017"):
    """用于量化的校准数据集"""
    for k, path in tqdm(enumerate(Path(data_dir).iterdir()), desc="Calibrate", unit="batch"):
        if k >= calibrate_num:
            break
        yield {ENV["input_name"]: preprocessing(path, **ENV)[1]}

## 构建简单 PyTorch 模型

In [3]:
from torch import nn


class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 1, 1, 0)
        self.hard_swish = nn.Hardswish()
        self.conv2 = nn.Conv2d(16, 16, 1, 1, 0, bias=False)

    def forward(self, x):
        x = self.conv1(x)
        x = self.hard_swish(x)
        x = self.conv2(x)
        return x

## 导出 ONNX 模型

In [4]:
import torch
from torch.onnx import OperatorExportTypes, utils

model = M()
model.eval()

shape = 1, 3, 6, 8
input_name = "data"
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
# model = torch.jit.trace(model, xx)
# 导出模型
output_name = "hard-swish-v1"
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=9,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

Exported graph: graph(%data : Float(1, 3, 6, 8, strides=[144, 48, 8, 1], requires_grad=0, device=cpu),
      %conv1.weight : Float(16, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu),
      %conv1.bias : Float(16, strides=[1], requires_grad=1, device=cpu),
      %conv2.weight : Float(16, 16, 1, 1, strides=[16, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv1/Conv_output_0 : Float(1, 16, 6, 8, strides=[768, 48, 8, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv1/Conv"](%data, %conv1.weight, %conv1.bias), scope: __main__.M::/torch.nn.modules.conv.Conv2d::conv1 # /media/pc/data/lxw/envs/anaconda3x/envs/py312/lib/python3.12/site-packages/torch/nn/modules/conv.py:456:0
  %/hard_swish/HardSigmoid_output_0 : Float(1, 16, 6, 8, strides=[768, 48, 8, 1], device=cpu) = onnx::HardSigmoid[alpha=0.16666666666666666, onnx_name="/hard_swish/HardSigmoid"](%/conv1/Conv_output_0), scope: __main

In [5]:
# output_name = "hard-swish-v2"
# utils.export(
#     model,               # torch 模型
#     xx,                         # 模型输入或者对于多个输入，使用元组
#     f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
#     export_params=True,        # 将训练后的参数权重存储在模型文件内
#     opset_version=14,          # 导出模型的 ONNX 版本
#     do_constant_folding=True,  # 是否执行常量折叠以进行优化
#     input_names = [input_name],    # 模型的输入名称
#     output_names = ['output'], # 模型的输出名称
#     keep_initializers_as_inputs=True,
#     # export_modules_as_functions=True,
#     verbose=True,
#     operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
#     # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
#     #               'output' : {0 : 'batch_size'}}
# )

In [6]:
# import numpy as np
# from tvm import relay

# data_np = (np.random.randint(0, 256, shape)/255).astype("float32")
# data_torch = torch.from_dlpack(data_np)

# model = M().eval()
# scripted_model = torch.jit.trace(model, data_torch).eval()
# shape_list = [(input_name, shape)]
# mod, params = relay.frontend.from_pytorch(scripted_model, shape_list)
# mod = relay.transform.InferType()(mod)

In [7]:
import onnx
from tvm import relay

onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx",)
mod, params = relay.frontend.from_onnx(onnx_model, {input_name: xx.shape}, freeze_params=True)
print(mod["main"])

fn (%data: Tensor[(1, 3, 6, 8), float32] /* ty=Tensor[(1, 3, 6, 8), float32] span=/conv1/Conv.data:0:0 */) -> Tensor[(1, 16, 6, 8), float32] {
  %0 = nn.conv2d(%data, meta[relay.Constant][0] /* ty=Tensor[(16, 3, 1, 1), float32] span=/conv1/Conv.conv1.weight:0:0 */, padding=[0, 0, 0, 0], channels=16, kernel_size=[1, 1]) /* ty=Tensor[(1, 16, 6, 8), float32] span=/conv1/Conv:0:0 */;
  %1 = nn.bias_add(%0, meta[relay.Constant][1] /* ty=Tensor[(16), float32] span=/conv1/Conv.conv1.bias:0:0 */) /* ty=Tensor[(1, 16, 6, 8), float32] span=/conv1/Conv:0:0 */;
  %2 = multiply(%1, 0.166667f /* ty=float32 span=/hard_swish/HardSigmoid:0:0 */) /* ty=Tensor[(1, 16, 6, 8), float32] span=/hard_swish/HardSigmoid:0:0 */;
  %3 = add(%2, 0.5f /* ty=float32 span=/hard_swish/HardSigmoid:0:0 */) /* ty=Tensor[(1, 16, 6, 8), float32] span=/hard_swish/HardSigmoid:0:0 */;
  %4 = clip(%3, a_min=0f, a_max=1f) /* ty=Tensor[(1, 16, 6, 8), float32] span=/hard_swish/HardSigmoid:0:0 */;
  %5 = multiply(%1, %4) /* ty=Tens

In [8]:
from tvm.relay.dataflow_pattern import (
    DFPatternCallback, 
    is_op, wildcard,
    is_constant,
    # is_tuple, 
    # is_tuple_get_item
)
from tvm import relay
from tvm.relay.dataflow_pattern import rewrite
from tvm_book.special.rewriter import simplify_hard_sigmoid, HardSwishSimplify
import tvm
from tvm_book.compiler.utils import merge_compiler


with tvm.transform.PassContext(opt_level=3):
    run_mod = simplify_hard_sigmoid(deepcopy(mod))
    run_mod["main"] = rewrite(HardSwishSimplify(), run_mod["main"])
    # run_mod = merge_compiler(run_mod, compiler_name="vta_special")

In [9]:
print(run_mod)

def @main(%data: Tensor[(1, 3, 6, 8), float32] /* ty=Tensor[(1, 3, 6, 8), float32] span=/conv1/Conv.data:0:0 */) -> Tensor[(1, 16, 6, 8), float32] {
  %0 = nn.conv2d(%data, meta[relay.Constant][0] /* ty=Tensor[(16, 3, 1, 1), float32] span=/conv1/Conv.conv1.weight:0:0 */, padding=[0, 0, 0, 0], channels=16, kernel_size=[1, 1]) /* ty=Tensor[(1, 16, 6, 8), float32] span=/conv1/Conv:0:0 */;
  %1 = nn.bias_add(%0, meta[relay.Constant][1] /* ty=Tensor[(16), float32] span=/conv1/Conv.conv1.bias:0:0 */) /* ty=Tensor[(1, 16, 6, 8), float32] span=/conv1/Conv:0:0 */;
  %2 = special.hard_swish(%1) /* ty=Tensor[(1, 16, 6, 8), float32] */;
  nn.conv2d(%2, meta[relay.Constant][2] /* ty=Tensor[(16, 16, 1, 1), float32] span=/conv2/Conv.conv2.weight:0:0 */, padding=[0, 0, 0, 0], channels=16, kernel_size=[1, 1]) /* ty=Tensor[(1, 16, 6, 8), float32] span=/conv2/Conv:0:0 */
}




In [10]:
# run_mod = deepcopy(mod)
with tvm.transform.PassContext(opt_level=3, disabled_pass={"AlterOpLayout"}):
    with relay.quantize.qconfig(
        calibrate_mode="percentile", weight_scale="max"):
        qmod = relay.quantize.quantize(run_mod, params, dataset=calibrateset())

InternalError: Traceback (most recent call last):
  2: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::topi::__mk_TVM1::{lambda(tvm::runtime::TVMArgs, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, tvm::runtime::TVMArgs, tvm::runtime::TVMRetValue*)
  1: tvm::topi::CommReduce(tvm::te::Tensor const&, tvm::runtime::Array<tvm::Integer, void> const&, std::function<tvm::PrimExpr (tvm::PrimExpr, tvm::runtime::Array<tvm::tir::IterVar, void> const&, tvm::runtime::Array<tvm::PrimExpr, void>, tvm::Span)>, bool, bool)
  0: tvm::topi::GetRealAxis(int, tvm::runtime::Array<tvm::Integer, void> const&)
  File "/media/pc/data/lxw/ai/tvm/include/tvm/topi/reduction.h", line 78
InternalError: Check failed: val < ndim (6 vs. 4) :  exceeds the maximum dimension 4

In [ ]:
qmod.show()